# Chapter 1: LLM Inputs ko Samajhna — Tokenization, Datasets, aur Embeddings

Is notebook mein hum seekhenge ki raw text data ko LLM ke liye kaise ready kiya jata hai. Text preprocessing pipeline ke major stages ye hain:
1. **Tokenization**: Raw text ko chote discrete pieces (tokens/words/characters) mein break karna.
2. **Vocabulary Building**: Un unique tokens ko unique numbers (Token IDs) mein map karna.
3. **BytePair Encoding (BPE)**: Modern LLMs (jaise GPT models) mein use hone wala advanced subword tokenizer.
4. **Sliding Window Data Sampling**: Model ko next token predict karne ke liye inputs aur target pairs prepare karna.
5. **Embeddings**: Token IDs aur unke position coordinates ko multi-dimensional vectors mein convert karna.

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/01.webp?timestamp=1" width="650px">

## 1. Installation aur Environment Setup

Sabse pehle hum neural networks ke liye **PyTorch** aur subword tokenization ke liye OpenAI ke **Tiktoken** library ko install karenge.

In [ ]:
# PyTorch aur Tiktoken ko chup-chaap (silently) install karne ke liye
!uv pip install torch tiktoken --quiet

In [ ]:
# Install hone ke baad versions verify karne ke liye import aur print karenge
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

## 2. Dataset load karna

Hum Edith Wharton ki short story *"The Verdict"* (public domain short story) load karenge aur use hum pure chapter mein text processing ke liye use karenge.

In [ ]:
# Story text file load karenge aur character stats display karenge
with open("./the-verdict.txt", "r", encoding='utf-8') as f:
    raw_text = f.read()

print("Total number of characters:", len(raw_text))
print("Preview first 100 characters:")
print(raw_text[:99])

## 3. Basic Tokenization

Sabse pehle hum simple regex-based tokenization dekhenge ki text ko spaces aur basic symbols par kaise split kiya jata hai.

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/04.webp" width="400px">

### 3.1 Punctuation aur Whitespace ke basis par splitting

In [ ]:
import re

text = "Hello, world. This, is a test."
# Comma (,), dot (.) aur whitespaces (\s) ke aaspas split karenge
# Parenthesis () capture group create karta hai, jisse delimiters bhi preserve ho jaate hain
result = re.split(r'([,.]|\s)', text)

print(result)

In [ ]:
# Output list mein se empty strings aur extra spaces ko clean kar rahe hain
result = [item for item in result if item.strip()]

print(result)

### 3.2 Advanced Punctuation Regex Splitting

In [ ]:
# Ab hum do double dashes (--) aur larger punctuations ko bhi separate tokens mein handle karenge
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]

print(result)

### 3.3 Pure Corpus (Book Text) ko Tokenize karna

In [ ]:
# Apne advanced punctuation regex ko hum poore Verdict story text par chalayenge
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]

print("First 30 tokens of preprocessed text:")
print(preprocessed[:30])
print("\nTotal tokens count:", len(preprocessed))

## 4. Tokens ko Token IDs (Numbers) mein badalna

LLM computational model hai, isliye hum har ek unique word/token ko integer number index (Token ID) se map karte hain, jise hum *Vocabulary* bolte hain.

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/06.webp" width="650px">

### 4.1 Vocabulary Construction

In [ ]:
# sorted(set(...)) se duplicates remove honge aur alphabetically unique tokens ki list milegi
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print("Vocabulary Size:", vocab_size)

In [ ]:
# Har unique token ko uske integer ID se map karne ke liye dictionary banayenge
vocab = {token:integer for integer, token in enumerate(all_words)}

In [ ]:
# Apni vocabulary map ke pehle 50 mappings ko verify kar rahe hain
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

### 4.2 Building a Simple Tokenizer (Version 1)

Hum ek clean Python class likhenge jiske do main methods honge: 
- `encode`: Text ko Token IDs (integers) mein convert karega.
- `decode`: Token IDs ko wapas human-readable text mein decode karega.

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/07.webp?123" width="650px">

In [ ]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        # Decoding ke liye ID se token check karne ke liye dictionary ko reverse kiya
        self.int_to_str = {i:s for s,i in vocab.items()}
    
    def encode(self, text):
        # Text preprocessing split
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        # Har word ko vocabulary ke integer index value se badlenge
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        # Integer IDs ko strings mein map karke space se join karenge
        text = " ".join([self.int_to_str[i] for i in ids])
        # Redundant spaces punctuations ke pehle se hatayenge
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [ ]:
# Apne SimpleTokenizerV1 ko book ke sample text string par instantiate aur test karenge
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""

print("Sample Text:")
print(text)

In [ ]:
# Sample text ko encode karke ids generate karenge
ids = tokenizer.encode(text)
print("Token IDs:", ids)

In [ ]:
# Decode method se check karenge ki wapas original text accurate display ho raha hai ya nahi
decoded_text = tokenizer.decode(ids)
print("Decoded Text:", decoded_text)

#### Out-of-Vocabulary (OOV) ki Problem

Agar hum V1 tokenizer ko koi naya word dein jo train corpus mein nahi tha (jaise 'Hello' ya 'tea'), toh hume `KeyError` dekhne ko milega kyunki woh words mapping dictionary mein present nahi hote.

In [ ]:
tokenizer = SimpleTokenizerV1(vocab)
text_oov = "Hello, do you like tea. Is this-- a test?"

# Niche di gayi line run karne par error aayega:
# tokenizer.encode(text_oov)

### 4.3 OOV Tokens aur Special Boundary Tokens (Version 2)

OOV problem ko solve karne ke liye hum vocabulary mein special markers introduce karenge:
- `<|unk|>`: Unknown tokens ko mark karne ke liye
- `<|endoftext|>`: Alag-alag documents ki sequence boundaries mark karne ke liye (BOS/EOS substitute in GPT-2)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/09.webp?123" width="600px">

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/10.webp" width="600px">

In [ ]:
# Vocabulary list mein special tokens daalkar mapping restructure kar rahe hain
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}

print("New Vocab size:", len(vocab.items()))

In [ ]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        # Agar token vocabulary mein missing hai, toh `<|unk|>` par fallback karenge
        preprocessed = [
            item if item in self.str_to_int 
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Punctuation formatting clean karenge
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [ ]:
# Tiny split tests: do alag text inputs ko special boundaries ke sath merge karke test karenge
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))

print(text)

In [ ]:
# '<|endoftext|>' ki integer ID vocabulary map mein verify karenge
print("ID of <|endoftext|>: ", vocab['<|endoftext|>'])

In [ ]:
# V2 tokenizer se encode chalayenge
encoded_ids = tokenizer.encode(text)
print("Encoded:", encoded_ids)

In [ ]:
# Decode to check output. Hello aur palace standard vocabulary mein missing the, isliye `<|unk|>` ban gaye
print("Decoded:", tokenizer.decode(encoded_ids))

### 4.4 Tokenization Visualizations

Tokens aur unke string representations ko clear aur colorful background format mein visualize karenge.

In [ ]:
from IPython.display import display, HTML

tokenizer = SimpleTokenizerV2(vocab)
token_ids = tokenizer.encode(text)
tokens = [tokenizer.decode([t]) for t in token_ids]

# Alag-alag colorful boxes display karne ke liye hex codes list
colors = [
    "#FFB3BA", "#FFDFBA", "#FFFFBA",
    "#BAFFC9", "#BAE1FF", "#D7BAFF"
]

html = ""
for i, token in enumerate(tokens):
    html += f"""
    <span style="
        background:{colors[i % len(colors)]};
        padding:4px 6px;
        margin:2px;
        color: #000;
        border-radius:4px;
        font-family:monospace;
    ">
    {repr(token)}
    </span>
    """

display(HTML(html))

## 5. BytePair Encoding (BPE)

BPE tokenizer raw strings ko direct space level par split karne ke bajaye frequency base par parts aur character tuples (subwords) banata hai. Isse out-of-vocabulary issue solve ho jata hai bina `<|unk|>` markers ke.

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/11.webp" width="400px">

### 5.1 Tiktoken library ka use GPT-2 split ke liye

In [ ]:
# Tiktoken imported library version print karke verify karenge
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

In [ ]:
# OpenAI GPT-2 ka primary pre-trained tokenizer load kar rahe hain
tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
# Kisi un-seen/unknown sequence ko encode aur decode karenge: subword split check ke liye
e1 = tokenizer.encode('jkbvhgjv')
print("Encoded subword list:", e1)

d1 = tokenizer.decode(e1)
print("Decoded string:", d1)

In [ ]:
# Special characters aur end-of-text tags preserve karte hue encoding check karenge
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces "
     "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print("Tiktoken Encoded Integers:", integers)

In [ ]:
# Wapas string decode karke print karenge
strings = tokenizer.decode(integers)
print("Tiktoken Decoded:", strings)

### 5.2 Pure Corpus (Story Book) ko BPE se Tokenize karna

In [ ]:
# File se story load karenge aur use GPT-2 standard tokenizer se encode karenge
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print("Total Tiktoken Tokens:", len(enc_text))

In [ ]:
# Verify karenge ki decode karne par original string length accurate reconstruct ho rahi hai
decoded_len = len(tokenizer.decode(enc_text))
print("Reconstructed Character Count:", decoded_len)

## 6. Sliding Window Technique se Data Sampling

Autoregressive model training ke liye hum tokenized sequences ko sliding windows mein break karte hain. Inputs aur targets create karne ke liye targets ko input se 1 token agey (right-shifted) rakhte hain.

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/12.webp" width="500px">

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/13.webp?123" width="650px">

### 6.1 Input-Target Pairs construct karna

In [ ]:
# Token arrays ka ek dummy slice process ke liye select kar rahe hain
enc_sample = enc_text[50:]

In [ ]:
context_size = 4

# Inputs (x) represent current window, targets (y) are shifted by 1 token
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x (Inputs): {x}")
print(f"y (Targets):      {y}")

In [ ]:
# Loop chalakar target tokens prediction steps ko index structure ke sath check karenge
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(context, "---->", desired)

In [ ]:
# Decode chalakar readable text mappings check karenge input aur target shifts ke liye
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

### 6.2 custom PyTorch Dataset aur DataLoader implement karna

Ab hum customized torch classes build karenge jo raw text data se batches handle karegi sliding offset calculation ke sath.

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/14.webp" width="650px">

In [ ]:
# PyTorch requirements import aur environment checks
import torch
print("PyTorch version:", torch.__version__)

In [ ]:
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Poore raw content ko encode karenge
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # Sliding window loop: overlap and stride sequence create karenge
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        # Total dataset items range return karenge
        return len(self.input_ids)

    def __getitem__(self, idx):
        # Index item data tensors retrieve karenge
        return self.input_ids[idx], self.target_ids[idx]

In [ ]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Base tokenizer selection
    tokenizer = tiktoken.get_encoding("gpt2")

    # custom GPTDataset instance trigger karenge
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Standard data pipeline structure loader create karenge
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [ ]:
# Python data iteration loop (next iterator) verify karne ke liye test check
# ex_iter = iter([1, 2, 3, 4])
# print(next(ex_iter))
# print(next(ex_iter))
# print(next(ex_iter))
# print(next(ex_iter))

In [ ]:
# DataLoader checks run: tiny sample batch verification check
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=3, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print("First batch output:", first_batch)

input_batch, target_batch = first_batch
print("Decoded Input:", tokenizer.decode(input_batch[0].tolist()))

In [ ]:
# Agle dataset batch ko evaluate karenge: overlap stride dynamics trace
second_batch = next(data_iter)
print("Second batch output:", second_batch)

input_batch, target_batch = second_batch
print("Decoded Input:", tokenizer.decode(input_batch[0].tolist()))
print("Decoded Test Array:", tokenizer.decode([2, 3, 5, 1]))

In [ ]:
# Subword BPE text splits me leading spacing handling behaviour verify
print(tokenizer.encode(" learn"))

In [ ]:
# Stride aur context overlapping check karne ke liye batch sizes 8 and stride 4 verify karenge
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
first_batch = next(data_iter)
input_batch, target_batch = first_batch

print("Inputs (Shape: [8, 4]):\n", input_batch)
print("Decoded input batch 0:", tokenizer.decode(input_batch[0].tolist()))

print("\nTargets (Shape: [8, 4]):\n", target_batch)
print("Decoded target batch 0:", tokenizer.decode(target_batch[0].tolist()))

## 7. Token Embeddings create karna

Token IDs ko low-dimensional representation key arrays se continuous multi-dimensional representation maps (dense vectors) mein build karenge PyTorch `nn.Embedding` matrix lookups use karke.

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/15.webp" width="500px">

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/16.webp?123" width="650px">

In [ ]:
input_ids = torch.tensor([2, 3, 5, 1])

# Tiny vocabulary size size limit 6 aur dense output dimensions coordinates size 3
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

print("Initial embedding layer weight matrix:")
print(embedding_layer.weight)

In [ ]:
# Specific weight indices retrieve (lookup matrix operations) index 3 aur index 2 coordinates ke liye
print("Embedding vector for ID 3:", embedding_layer(torch.tensor([3])))
print("Embedding vector for ID 2:", embedding_layer(torch.tensor([2])))

In [ ]:
# Multi-token index elements parallel lookup checks
print("Embedding output shape:", embedding_layer(input_ids).shape)
print(embedding_layer(input_ids))

## 8. Sequence Positions Encode karna (Positional Embeddings)

Transformer base Attention logic input sequence orders ko manually evaluate nahi karta. Is sequence sequence direction flow track karne ke liye hum token representations maps mein sequence index location offsets inject karte hain absolute position mappings ke sath.

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/17.webp" width="500px">

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/18.webp" width="650px">

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/19.webp" width="500px">

In [ ]:
# Token embedding settings parameter definition checks
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(token_embedding_layer)

In [ ]:
# Token embedding parameters shapes aur stats details checks
print("Token embedding layer:", token_embedding_layer)
print("Weight tensor shape:", token_embedding_layer.weight.shape)
print(token_embedding_layer.weight)

In [ ]:
# Test check runs parameters define, inputs select mapping dataloader loops use karenge
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [ ]:
# Dataloader outputs targets inspect checks
print("Token IDs:\n", inputs)
print("\nInputs shape:", inputs.shape)

In [ ]:
# Token ids ko unke base embeddings format vectors par map karenge
token_embeddings = token_embedding_layer(inputs)
print("Token embeddings shape:", token_embeddings.shape)
print(token_embeddings)

In [ ]:
# Positional offsets mapping create karenge matching sequence coordinates context range
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

print("Positional embedding weights:")
print(pos_embedding_layer.weight)

In [ ]:
# 0 se context maximum offset values positional embeds mapping trigger checks
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print("Positional embeddings shape:", pos_embeddings.shape)
print(pos_embeddings)

In [ ]:
# Embedding dense inputs aur position index mappings sum coordinate representation generate karenge
input_embeddings = token_embeddings + pos_embeddings
print("Final input embeddings shape:", input_embeddings.shape)
print(input_embeddings)